In [ ]:
%idle_timeout 2880 # Tiempo máximo de inactividad
%glue_version 5.0 # Versión de AWS Glue
%worker_type G.1X # Tipo worker
%number_of_workers 5 # Número de workers para Spark

# Importamos las librerías necesarias
import sys
from pyspark.sql import SparkSession, Window
import pyspark.sql.functions as F
from awsglue.context import GlueContext
from pyspark.context import SparkContext
from awsglue.job import Job
import pandas as pd

SILVER_BUCKET = "YOUR_SILVER_BUCKET"
GOLD_BUCKET = "YOUR_GOLD_BUCKET"

sc = SparkContext.getOrCreate() # Obtenemos/Creamos el SparkContext
glueContext = GlueContext(sc) # Contexto de Glue
spark = glueContext.spark_session # Sesión de Spark
job = Job(glueContext) # Creamos el job de Glue

dfs = [] # Lista para almacenar DataFrames por año
for i in range(2, 6):
    year = 2020 + i

    year_path = (
        f"s3://{SILVER_BUCKET}/"
        f"LINKUSD/year={year}/*"
    )

    temp_df = spark.read.parquet(year_path).withColumn("year", F.lit(2020 + i)) # Leemos en formato parquet y añadimos la columna year
    dfs.append(temp_df)

df = dfs[0]
for next_df in dfs[1:]:
    df = df.unionByName(next_df) # Unimos los DataFrames en uno solo

df = df.withColumn("_id", F.monotonically_increasing_id()) # Añadimos una identificador a cada fila
window_spec = Window.orderBy("year", "_id")

# SMA 200
df = df.withColumn("SMA_200", F.avg("close").over(window_spec.rowsBetween(-199, 0))) # Calculamos un SMA de 200 días

# EMA 50
# Convert to Pandas for recursive exponential moving averages
pdf = df.orderBy("timestamp").toPandas()

# Make sure close is numeric
pdf["close"] = pd.to_numeric(pdf["close"], errors="coerce")

# EMA-50
pdf["EMA_50"] = pdf["close"].ewm(
    span=50,
    adjust=False
).mean()

# EMA-12 and EMA-26
pdf["EMA_12"] = pdf["close"].ewm(
    span=12,
    adjust=False
).mean()

pdf["EMA_26"] = pdf["close"].ewm(
    span=26,
    adjust=False
).mean()

# MACD
pdf["MACD"] = (
    pdf["EMA_12"] - pdf["EMA_26"]
)

# Convert back to Spark DataFrame
df = spark.createDataFrame(pdf)

# RSI
df = df.withColumn("diff", F.col("close") - F.lag("close", 1).over(window_spec)) # Calculamos RSI, basado en ganancias y pérdidas promedio
df = df.withColumn("gain", F.when(F.col("diff") > 0, F.col("diff")).otherwise(0))
df = df.withColumn("loss", F.when(F.col("diff") < 0, F.abs(F.col("diff"))).otherwise(0))
avg_gain = F.avg("gain").over(window_spec.rowsBetween(-13, 0)) # Promedio de ganancias y pérdidas en 14 periodos
avg_loss = F.avg("loss").over(window_spec.rowsBetween(-13, 0))
df = df.withColumn("RSI", 100 - (100 / (1 + (avg_gain / (avg_loss + 0.00001))))) # Cálculo final de RSI

cols_finales = ["symbol", "open", "high", "low", "close", "volume", "year", "SMA_200", "EMA_50", "MACD", "RSI"] # Seleccionamos las columnas finales
df_final = df.select(*cols_finales)

df_final.write \
    .mode("overwrite") \
    .partitionBy("year") \
    .parquet(
    f"s3://{GOLD_BUCKET}/LINKUSD/"
     ) # Guardamos el DataFrame final en S3 en formato parquet, en linkusd-bucket-oro

job.commit()

In [ ]:
df_final.show()